# Day 05 下午学生项目：电商用户多维分析

**小组编号：** 03  
**成员：** 张智超、徐凯昊、任嘉硕、黎容舟、龙俊甫  
**专题方向：** A：用户生命周期

> 请只在标有 `步骤` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "03"
MEMBERS = ["张智超", "徐凯昊", "任嘉硕", "黎容舟", "龙俊甫"]
TOPIC = "A：用户生命周期"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_data_path(start=None):
    start = Path.cwd() if start is None else Path(start)
    roots = [start, *start.parents]
    names = [Path("output/day04_project/ecommerce_customer_cleaned.csv")]
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.exists():
                return candidate.resolve()
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

DATA_PATH = find_data_path()
ROOT = DATA_PATH.parents[2] if DATA_PATH.name == "ecommerce_customer_cleaned.csv" and DATA_PATH.parent.name == "day04_project" else DATA_PATH.parent
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH.relative_to(ROOT))
print("输出：", OUTPUT_DIR.relative_to(ROOT))

小组： 03 ['张智超', '徐凯昊', '任嘉硕', '黎容舟', '龙俊甫']
专题： A：用户生命周期
输入： output/day04_project/ecommerce_customer_cleaned.csv
输出： output/day05_analysis


### 检查点0

- [x] 已填写组号、成员和专题；
- [x] Notebook已按仓库统一文件名保存；
- [x] 输出目录已统一为`output/day05_analysis/`。

## 任务1：加载并验收数据（必做）

In [2]:
# 步骤 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)
raw_shape = df.shape

# 步骤 2：输出shape、前5行和字段类型
print("数据形状：", raw_shape)
display(df.head())
print(df.dtypes)

# 步骤 3：计算以下验收结果
core_fields = [
    "CustomerID", "Churn", "Tenure", "OrderCount", "CouponUsed",
    "CashbackAmount", "HourSpendOnApp", "SatisfactionScore",
    "DaySinceLastOrder", "Complain"
]
validation = {
    "行数": raw_shape[0],
    "列数": raw_shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_fields].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}
validation

tenure_order = ["0-3个月", "4-6个月", "7-12个月", "13-24个月", "25个月及以上"]
df["TenureGroup"] = pd.cut(
    df["Tenure"],
    bins=[-1, 3, 6, 12, 24, np.inf],
    labels=tenure_order,
    ordered=True,
)

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,0-6个月,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,0-6个月,1


CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
TenureGroup                     object
IsMobileLogin                    int64
dtype: object


In [3]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert validation["行数"] == 5630, "数据行数应为5630"
assert validation["列数"] == 22, "数据列数应为22"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert validation["核心字段缺失数"] == 0, "核心字段不应缺失"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("检查点1通过")

检查点1通过


**数据粒度：** 一行代表一名用户，CustomerID是用户唯一标识，OrderCount表示该用户的订单次数。

## 任务2：公共基础指标（必做）

In [4]:
# 步骤：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数

overall_metrics = pd.DataFrame({
    "指标": [
        "总用户数", "流失人数", "总体流失率", "平均订单数", "订单数中位数",
        "平均优惠券使用次数", "平均返现金额", "平均App使用时长",
        "平均满意度", "平均距上次下单天数"
    ],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})

display(overall_metrics)

,指标,数值
0,总用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现金额,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [5]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# 步骤：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, "总体流失率不正确"
print("检查点2通过")

检查点2通过


## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [6]:
# 步骤：填写你的分组字段
segment_field = "TenureGroup"

# 步骤：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field, observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均返现=("CashbackAmount", "mean"),
    )
    .reset_index()
)
segment_analysis["用户占比"] = segment_analysis["用户数"] / df["CustomerID"].nunique()

# 步骤：重置索引、排序并展示
segment_analysis[segment_field] = pd.Categorical(
    segment_analysis[segment_field],
    categories=tenure_order,
    ordered=True,
)
segment_analysis = segment_analysis.sort_values(segment_field).reset_index(drop=True)
display(segment_analysis)

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,用户占比
0,0-3个月,1560,653,0.42,2.32,155.66,0.28
1,4-6个月,590,44,0.07,2.97,169.91,0.10
2,7-12个月,1584,156,0.10,2.75,163.31,0.28
3,13-24个月,1467,95,0.06,3.70,204.92,0.26
4,25个月及以上,429,0,0.00,3.55,222.34,0.08


In [7]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

检查点3通过


### 专题分析记录

**数据现象：** 0—3个月用户的流失率为41.86%，明显高于其他生命周期组；13—24个月用户的平均订单数为3.70，高于0—3个月用户的2.32。

**可能解释：** 新用户尚未形成稳定使用习惯，可能更容易流失；生命周期较长的用户可能已形成相对稳定的购买行为，但这一结果仍需结合营销触达、服务记录和用户来源进一步验证。

## 任务4：双维度交叉分析（必做）

In [8]:
# 步骤：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "Complain"

# 步骤：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
    )
    .reset_index()
)

# 步骤：新增“样本提示”列；用户数<30标记为“小样本”，否则为“可观察”
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")

# 步骤：按流失率或用户数排序并展示
cross_analysis[dim_1] = pd.Categorical(
    cross_analysis[dim_1],
    categories=tenure_order,
    ordered=True,
)
cross_analysis = cross_analysis.sort_values(["流失率", "用户数"], ascending=[False, False]).reset_index(drop=True)
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示
0,0-3个月,1,522,345,0.66,2.55,可观察
1,0-3个月,0,1038,308,0.30,2.20,可观察
2,4-6个月,1,137,30,0.22,3.03,可观察
3,7-12个月,1,406,81,0.20,2.67,可观察
4,13-24个月,1,414,52,0.13,3.35,可观察
5,7-12个月,0,1178,75,0.06,2.78,可观察
6,13-24个月,0,1053,43,0.04,3.85,可观察
7,4-6个月,0,453,14,0.03,2.95,可观察
8,25个月及以上,0,304,0,0.00,3.75,可观察
9,25个月及以上,1,125,0,0.00,3.06,可观察


In [9]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的组合：** 生命周期为0—3个月且发生投诉的用户。

**该组合的样本量与流失率：** 该组合共有522名用户，流失率为66.09%；同一生命周期内未投诉用户共有1038人，流失率为29.67%。

**为什么不能直接下因果结论：** 当前数据属于用户层面的观察数据，投诉与流失之间可能同时受到服务体验、用户预期和用户活跃度等因素影响，因此只能说明二者存在关联，不能证明投诉直接导致流失。

## 任务5：报表输出与回读验证（必做）

In [10]:
# 步骤：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# 步骤：循环导出并重新读取；打印每个文件的shape
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    print(filename, reloaded.shape)

overall_metrics.csv (10, 2)
segment_analysis.csv (5, 7)
cross_analysis.csv (10, 7)


In [11]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

检查点5通过：三个CSV均已成功导出并回读。


## 任务6：结论、限制与建议（必做）

### 结论1

在0—3个月用户中，流失率为41.86%，与13—24个月用户的6.48%相比高35.38个百分点。对应证据表：segment_analysis.csv。

### 结论2

在13—24个月用户中，平均订单数为3.70，与0—3个月用户的2.32相比高1.39次。当前样本表明生命周期与订单行为存在关联，仍需结合用户来源和活动参与数据进一步验证。对应证据表：segment_analysis.csv。

### 结论3

在0—3个月且发生投诉的用户中，流失率为66.09%，与同一生命周期内未投诉用户的29.67%相比高36.42个百分点。当前样本表明投诉与流失存在关联，但不能据此判断投诉直接导致流失。对应证据表：cross_analysis.csv。

### 分析限制

数据缺少订单金额、订单日期、营销触达和投诉处理结果，因此不能计算销售额、客单价和时间趋势，也无法仅凭当前观察数据识别因果关系。25个月及以上用户的流失率为0，也需要进一步核查样本筛选、标签定义和观察窗口。

### 运营建议与验证方式

可以优先关注0—3个月且发生投诉的用户，设置投诉后的回访、补救和新用户关怀方案。实施时应采用分组对照或A/B测试，并补充投诉原因、处理时长、营销触达和后续订单数据，用于验证该方案是否能够改善留存。

## 拓展任务（选做）

In [12]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# 选做
activity_cut = pd.qcut(df["OrderCount"], q=4, duplicates="drop")
activity_level_count = len(activity_cut.cat.categories)
activity_label_sets = {
    2: ["低活跃", "高活跃"],
    3: ["低活跃", "中活跃", "高活跃"],
    4: ["低活跃", "中低活跃", "中高活跃", "高活跃"],
}
activity_labels = activity_label_sets.get(
    activity_level_count,
    [f"活跃层级{i + 1}" for i in range(activity_level_count)],
)
activity_ranges = dict(zip(activity_labels, activity_cut.cat.categories.astype(str)))
df["OrderActivityGroup"] = activity_cut.cat.rename_categories(activity_labels)

order_activity_analysis = (
    df.groupby("OrderActivityGroup", observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均优惠券数=("CouponUsed", "mean"),
        平均返现=("CashbackAmount", "mean"),
    )
    .reset_index()
)
order_activity_analysis["订单范围"] = (
    order_activity_analysis["OrderActivityGroup"].astype(str).map(activity_ranges)
)
order_activity_analysis["用户占比"] = (
    order_activity_analysis["用户数"] / df["CustomerID"].nunique()
)
order_activity_analysis = order_activity_analysis[
    [
        "OrderActivityGroup", "订单范围", "用户数", "用户占比",
        "流失人数", "流失率", "平均订单数", "平均优惠券数", "平均返现",
    ]
]

z = 1.96
cross_analysis_enhanced = cross_analysis.copy()
n = cross_analysis_enhanced["用户数"].astype(float)
p_hat = cross_analysis_enhanced["流失率"].astype(float)
denominator = 1 + z ** 2 / n
center = (p_hat + z ** 2 / (2 * n)) / denominator
margin = z * np.sqrt(
    (p_hat * (1 - p_hat) + z ** 2 / (4 * n)) / n
) / denominator
cross_analysis_enhanced["流失率95%CI下限"] = (center - margin).clip(lower=0)
cross_analysis_enhanced["流失率95%CI上限"] = (center + margin).clip(upper=1)
cross_analysis_enhanced["严格样本提示"] = np.where(
    n < 50, "谨慎解释", "样本充足"
)

life_long = segment_analysis.copy()
life_long["TenureGroup"] = life_long["TenureGroup"].astype(str)
life_long = life_long.melt(
    id_vars=["TenureGroup", "用户数"],
    value_vars=["流失率", "平均订单数", "平均返现"],
    var_name="指标",
    value_name="数值",
)
life_long = life_long.rename(columns={"TenureGroup": "分组值"})
life_long["分析主题"] = "用户生命周期"
life_long["分组字段"] = "TenureGroup"

activity_long = order_activity_analysis.copy()
activity_long["OrderActivityGroup"] = activity_long["OrderActivityGroup"].astype(str)
activity_long = activity_long.melt(
    id_vars=["OrderActivityGroup", "用户数"],
    value_vars=["流失率", "平均订单数", "平均优惠券数", "平均返现"],
    var_name="指标",
    value_name="数值",
)
activity_long = activity_long.rename(columns={"OrderActivityGroup": "分组值"})
activity_long["分析主题"] = "订单活跃度"
activity_long["分组字段"] = "OrderActivityGroup"

plotting_long_table = pd.concat([life_long, activity_long], ignore_index=True)
plotting_long_table = plotting_long_table[
    ["分析主题", "分组字段", "分组值", "指标", "数值", "用户数"]
]

long_term = df[df["Tenure"] >= 25].copy()
long_term["长期用户细分"] = pd.cut(
    long_term["Tenure"],
    bins=[24, 36, 48, np.inf],
    labels=["25-36个月", "37-48个月", "49个月及以上"],
    ordered=True,
)
long_term_detail = (
    long_term.groupby("长期用户细分", observed=True)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
    )
    .reset_index()
)
long_term_summary = "；".join(
    f"{row['长期用户细分']}：{int(row['用户数'])}人，流失{int(row['流失人数'])}人"
    for _, row in long_term_detail.iterrows()
)
counterintuitive_checks = pd.DataFrame(
    [
        {
            "反直觉结果": "25个月及以上用户流失率为0",
            "核查方案": "标签与缺失核查",
            "具体操作": "检查长期用户的Churn取值、缺失数、用户数和流失人数",
            "当前结果": (
                f"用户数{long_term['CustomerID'].nunique()}，"
                f"Churn缺失{long_term['Churn'].isna().sum()}，"
                f"流失人数{int(long_term['Churn'].sum())}"
            ),
            "后续需求": "核对流失标签定义、观察窗口和清洗前后记录",
        },
        {
            "反直觉结果": "25个月及以上用户流失率为0",
            "核查方案": "长期用户再分层核查",
            "具体操作": "将长期用户按使用月数继续分层并比较各层样本量和流失人数",
            "当前结果": long_term_summary,
            "后续需求": "扩大时间窗口并检查是否存在幸存者偏差",
        },
    ]
)

optional_outputs = {
    "order_activity_analysis.csv": order_activity_analysis,
    "cross_analysis_enhanced.csv": cross_analysis_enhanced,
    "plotting_long_table.csv": plotting_long_table,
    "counterintuitive_checks.csv": counterintuitive_checks,
}
for filename, table in optional_outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"

assert len(order_activity_analysis) >= 2
assert {"流失率95%CI下限", "流失率95%CI上限", "严格样本提示"}.issubset(
    cross_analysis_enhanced.columns
)
assert {"分析主题", "分组字段", "分组值", "指标", "数值", "用户数"}.issubset(
    plotting_long_table.columns
)
assert len(counterintuitive_checks) >= 2

display(order_activity_analysis)
display(cross_analysis_enhanced)
display(plotting_long_table)
display(counterintuitive_checks)
print("拓展任务检查通过")

,OrderActivityGroup,订单范围,用户数,用户占比,流失人数,流失率,平均订单数,平均优惠券数,平均返现
0,低活跃,"(0.999, 2.0]",4034,0.72,704,0.17,1.57,1.10,170.77
1,中活跃,"(2.0, 3.0]",371,0.07,68,0.18,3.00,1.91,174.60
2,高活跃,"(3.0, 16.0]",1225,0.22,176,0.14,7.55,3.68,199.26


,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,样本提示,流失率95%CI下限,流失率95%CI上限,严格样本提示
0,0-3个月,1,522,345,0.66,2.55,可观察,0.62,0.70,样本充足
1,0-3个月,0,1038,308,0.30,2.20,可观察,0.27,0.33,样本充足
2,4-6个月,1,137,30,0.22,3.03,可观察,0.16,0.30,样本充足
3,7-12个月,1,406,81,0.20,2.67,可观察,0.16,0.24,样本充足
4,13-24个月,1,414,52,0.13,3.35,可观察,0.10,0.16,样本充足
5,7-12个月,0,1178,75,0.06,2.78,可观察,0.05,0.08,样本充足
6,13-24个月,0,1053,43,0.04,3.85,可观察,0.03,0.05,样本充足
7,4-6个月,0,453,14,0.03,2.95,可观察,0.02,0.05,样本充足
8,25个月及以上,0,304,0,0.00,3.75,可观察,0.00,0.01,样本充足
9,25个月及以上,1,125,0,0.00,3.06,可观察,0.00,0.03,样本充足


,分析主题,分组字段,分组值,指标,数值,用户数
0,用户生命周期,TenureGroup,0-3个月,流失率,0.42,1560
1,用户生命周期,TenureGroup,4-6个月,流失率,0.07,590
2,用户生命周期,TenureGroup,7-12个月,流失率,0.10,1584
3,用户生命周期,TenureGroup,13-24个月,流失率,0.06,1467
4,用户生命周期,TenureGroup,25个月及以上,流失率,0.00,429
5,用户生命周期,TenureGroup,0-3个月,平均订单数,2.32,1560
6,用户生命周期,TenureGroup,4-6个月,平均订单数,2.97,590
7,用户生命周期,TenureGroup,7-12个月,平均订单数,2.75,1584
8,用户生命周期,TenureGroup,13-24个月,平均订单数,3.70,1467
9,用户生命周期,TenureGroup,25个月及以上,平均订单数,3.55,429


,反直觉结果,核查方案,具体操作,当前结果,后续需求
0,25个月及以上用户流失率为0,标签与缺失核查,检查长期用户的Churn取值、缺失数、用户数和流失人数,用户数429，Churn缺失0，流失人数0,核对流失标签定义、观察窗口和清洗前后记录
1,25个月及以上用户流失率为0,长期用户再分层核查,将长期用户按使用月数继续分层并比较各层样本量和流失人数,25-36个月：425人，流失0人；49个月及以上：4人，流失0人,扩大时间窗口并检查是否存在幸存者偏差


拓展任务检查通过


## 提交前检查

- [x] 已填写组号、成员和专题；
- [x] 已重启内核并从头运行成功；
- [x] 所有比例表都包含样本量；
- [x] 三个CSV已导出并回读；
- [x] 至少3条结论可对应到具体表格；
- [x] 已写明分析限制和验证建议；
- [x] 没有把返现写成消费额，没有把相关写成因果。